# AgriNexus AI — Production Disease Detection Model Training (Google Colab)
This notebook resumes the Three-Phase training of **ConvNeXt-Base (61 classes)** on an NVIDIA T4 GPU.

### Prerequisites:
1. Go to **Runtime → Change runtime type** and set Hardware Accelerator to **T4 GPU**.
2. Upload `agrinexus_training_code.zip` and `agrinexus_dataset.zip` to the Files panel on the left (or via Google Drive).

In [ ]:
# 1. Verify NVIDIA T4 GPU
!nvidia-smi

In [ ]:
# 2. Install Required Production ML Libraries
!pip install -q "albumentations>=1.4.0" "torchmetrics>=1.4.0" "timm>=1.0.3" "scikit-learn>=1.5.0" pyyaml pillow tqdm

In [ ]:
# 3. Unzip Code and Dataset Archives
import os

if os.path.exists('agrinexus_training_code.zip'):
    print("Unpacking code and checkpoints...")
    !unzip -q -o agrinexus_training_code.zip

if os.path.exists('agrinexus_dataset.zip'):
    print("Unpacking dataset (this may take ~1-2 minutes)...")
    !unzip -q -o agrinexus_dataset.zip

print("Directory structure verified:")
!ls -la ml/disease/checkpoints/
!ls -la sample\ images/

In [ ]:
# 4. Resume Phase 1 Training on CUDA
# Phase 0 (5 epochs, 90.19% acc) is already complete in last_epoch.pth.
# This starts Phase 1 (Epoch 6) with Stages 3 & 4 unfrozen and live tqdm progress.
!python ml/disease/train.py \
  --config ml/disease/config.yaml \
  --resume ml/disease/checkpoints/last_epoch.pth \
  --device cuda

In [ ]:
# 5. Run Evaluation Harness on PlantDoc & OOD Distractor Set
!python ml/disease/evaluate.py \
  --checkpoint ml/disease/checkpoints/best_model.pth \
  --gate1-checkpoint ml/disease/checkpoints/gate1_model.pth \
  --config ml/disease/config.yaml \
  --device cuda

In [ ]:
# 6. Download the Trained Best Model and Logs
from google.colab import files

if os.path.exists('ml/disease/checkpoints/best_model.pth'):
    files.download('ml/disease/checkpoints/best_model.pth')
if os.path.exists('ml/disease/checkpoints/epoch_log.csv'):
    files.download('ml/disease/checkpoints/epoch_log.csv')